
# RANDOM FOREST – MỨC 5: NESTED K-FOLD CROSS-VALIDATION

## Mục tiêu
- Sử dụng dữ liệu Diamonds Prices2022 đã được làm sạch.
- Mã hóa các thuộc tính phân loại bằng OrdinalEncoder.
- Mở rộng GridSearchCV từ 36 lên 90 cấu hình siêu tham số.
- Sử dụng Nested K-Fold gồm 5 Fold bên ngoài và 3 Fold bên trong.
- Đánh giá mô hình bằng MSE, RMSE, MAE và R².
- So sánh kết quả với mức 3 và mức 4 K-Fold.

## Phương pháp
- Inner K-Fold: tìm bộ siêu tham số tốt nhất.
- Outer K-Fold: đánh giá mô hình trên dữ liệu chưa tham gia quá trình huấn luyện và chọn tham số.
- Kết quả cuối cùng là trung bình và độ lệch chuẩn của 5 Outer Fold.


In [1]:

# Thu vien xu ly duong dan
from pathlib import Path

# Thu vien do thoi gian thuc thi
import time

# Thu vien xu ly du lieu dang bang
import pandas as pd

# Thu vien tinh toan so hoc
import numpy as np

# Thu vien chia du lieu thanh cac Fold
from sklearn.model_selection import KFold

# Thu vien tim kiem sieu tham so
from sklearn.model_selection import GridSearchCV

# Thu vien ma hoa du lieu phan loai
from sklearn.preprocessing import OrdinalEncoder

# Thu vien tien xu ly cac cot du lieu
from sklearn.compose import ColumnTransformer

# Thu vien ket hop cac buoc xu ly va mo hinh
from sklearn.pipeline import Pipeline

# Mo hinh hoi quy Random Forest
from sklearn.ensemble import RandomForestRegressor

# Cac chi so danh gia mo hinh hoi quy
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

# Kiem tra da nap thu vien thanh cong
print("Da nap day du thu vien.")

# Duong dan khi chay tu experiments/random_forest
file_path = Path("../../data/raw/Diamonds Prices2022.csv")

# Thu duong dan tu thu muc goc du an
if not file_path.exists():
    file_path = Path("data/raw/Diamonds Prices2022.csv")

# Bao loi neu khong tim thay tep du lieu
if not file_path.exists():
    raise FileNotFoundError("Khong tim thay file Diamonds Prices2022.csv")

# Doc CSV, dung cot dau tien lam chi so dong
diamonds = pd.read_csv(file_path, index_col=0)

# Hien thi duong dan dang su dung
print("Duong dan:", file_path.resolve())

# Hien thi kich thuoc du lieu
print("Kich thuoc du lieu:", diamonds.shape)

# Dem tong so gia tri bi thieu
print("So gia tri thieu:", diamonds.isnull().sum().sum())

# Dem so dong trung lap
print("So dong trung lap:", diamonds.duplicated().sum())

# Dem so dong co it nhat mot kich thuoc khong hop le
invalid_size = (
    (diamonds["x"] <= 0) |
    (diamonds["y"] <= 0) |
    (diamonds["z"] <= 0)
)

# Hien thi so dong co kich thuoc khong hop le
print("So dong kich thuoc khong hop le:", invalid_size.sum())

# Hien thi 5 dong dau tien
display(diamonds.head())


Da nap day du thu vien.
Duong dan: D:\DiamondPrediction\data\raw\Diamonds Prices2022.csv
Kich thuoc du lieu: (53943, 10)
So gia tri thieu: 0
So dong trung lap: 149
So dong kich thuoc khong hop le: 20


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75



## 2. Làm sạch dữ liệu

Thực hiện các bước:
1. Tạo bản sao dữ liệu gốc.
2. Xóa các dòng chứa giá trị thiếu.
3. Loại bỏ những dòng có x, y hoặc z nhỏ hơn hoặc bằng 0.
4. Xóa các dòng trùng lặp.
5. Đặt lại chỉ số dòng.

Sử dụng cùng quy trình làm sạch như mức 4 để đảm bảo tính nhất quán khi so sánh.


In [2]:

# Tao ban sao de khong thay doi du lieu goc
df = diamonds.copy()

# Luu so dong ban dau
so_dong_ban_dau = len(df)

# Xoa cac dong co gia tri thieu
df = df.dropna()

# Chi giu nhung dong co x, y, z lon hon 0
df = df[
    (df["x"] > 0) &
    (df["y"] > 0) &
    (df["z"] > 0)
]

# Xoa nhung dong trung lap
df = df.drop_duplicates()

# Dat lai chi so dong tu 0
df = df.reset_index(drop=True)

# Hien thi so dong ban dau
print("So dong ban dau:", so_dong_ban_dau)

# Hien thi so dong sau khi lam sach
print("So dong sau lam sach:", len(df))

# Tinh so dong da loai bo
print("So dong bi loai:", so_dong_ban_dau - len(df))

# Hien thi kich thuoc du lieu sau lam sach
print("Kich thuoc sau lam sach:", df.shape)

# Hien thi 5 dong dau tien
display(df.head())


So dong ban dau: 53943
So dong sau lam sach: 53775
So dong bi loai: 168
Kich thuoc sau lam sach: (53775, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75



## 3. Xác định đặc trưng và biến mục tiêu

- X: các thuộc tính dùng để dự đoán giá kim cương.
- y: cột price, là giá kim cương cần dự đoán.

Không đưa price vào X để tránh rò rỉ dữ liệu.

Dữ liệu chưa được mã hóa tại bước này. OrdinalEncoder sẽ nằm bên trong Pipeline và được học lại trên tập Train của từng Fold.


In [3]:

# Lay tat ca cac cot tru price lam dac trung
X = df.drop(columns=["price"])

# Lay cot price lam bien muc tieu
y = df["price"]

# Hien thi kich thuoc cua X
print("Kich thuoc X:", X.shape)

# Hien thi kich thuoc cua y
print("Kich thuoc y:", y.shape)

# Hien thi danh sach dac trung
print("Cac dac trung:", X.columns.tolist())

# Hien thi ten bien muc tieu
print("Bien muc tieu:", y.name)


Kich thuoc X: (53775, 9)
Kich thuoc y: (53775,)
Cac dac trung: ['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z']
Bien muc tieu: price



## 4. Xây dựng Pipeline

Pipeline gồm hai bước:

1. Preprocessor:
   - Mã hóa color, clarity và cut bằng OrdinalEncoder.
   - Giữ nguyên các thuộc tính số.

2. Model:
   - RandomForestRegressor.

Đặt bước mã hóa bên trong Pipeline để tránh việc bộ mã hóa học thông tin từ Fold kiểm tra.

Các thuộc tính phân loại được mã hóa theo thứ tự chất lượng đã xác định.


In [4]:

# Thu tu mau sac tu J den D
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Thu tu do tinh khiet tu I1 den IF
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Thu tu chat luong giac cat
cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]

# Danh sach cac cot can ma hoa
categorical_cols = ["color", "clarity", "cut"]

# Khoi tao bo ma hoa theo thu tu da xac dinh
encoder = OrdinalEncoder(
    categories=[color_order, clarity_order, cut_order]
)

# Tao bo tien xu ly du lieu
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", encoder, categorical_cols)
    ],
    remainder="passthrough"
)

# Khoi tao mo hinh Random Forest
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=1
)

# Ket hop tien xu ly va mo hinh trong Pipeline
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf)
    ]
)

# Hien thi cau truc Pipeline
print(pipeline)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categorical',
                                                  OrdinalEncoder(categories=[['J',
                                                                              'I',
                                                                              'H',
                                                                              'G',
                                                                              'F',
                                                                              'E',
                                                                              'D'],
                                                                             ['I1',
                                                                              'SI2',
                                                                              'SI1',
      


## 5. Thiết lập Nested K-Fold và GridSearchCV

Sử dụng hai vòng Cross-validation:

### Outer K-Fold
- Số Fold: 5.
- Dùng để đánh giá mô hình.
- Mỗi lần lấy 4 Fold làm Train và 1 Fold làm Test.

### Inner K-Fold
- Số Fold: 3.
- Chỉ chạy trên tập Train của Outer Fold.
- Dùng để lựa chọn siêu tham số bằng GridSearchCV.

### Không gian tìm kiếm mức 5
- n_estimators: 100, 150, 200, 250, 300.
- max_depth: None, 20, 30.
- min_samples_leaf: 1, 2.
- max_features: 0.6, 0.8, 1.0.

Tổng cộng 90 cấu hình.

GridSearchCV sử dụng RMSE làm tiêu chí lựa chọn, với scoring = neg_root_mean_squared_error.


In [5]:

# Tao 5 Fold ben ngoai de danh gia mo hinh
outer_kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Tao 3 Fold ben trong de tim sieu tham so
inner_kf = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Khai bao khong gian tim kiem muc 5
param_grid = {
    "model__n_estimators": [100, 150, 200, 250, 300],
    "model__max_depth": [None, 20, 30],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": [0.6, 0.8, 1.0]
}

# Tinh tong so to hop sieu tham so
so_cau_hinh = int(
    np.prod([len(values) for values in param_grid.values()])
)

# Tinh so luot huan luyen Cross-validation
so_luot_cv = so_cau_hinh * 3 * 5

# Hien thi so cau hinh
print("So cau hinh:", so_cau_hinh)

# Hien thi so Fold ben ngoai
print("Outer Fold:", outer_kf.get_n_splits())

# Hien thi so Fold ben trong
print("Inner Fold:", inner_kf.get_n_splits())

# Hien thi tong so luot fit CV
print("Tong so luot fit CV:", so_luot_cv)


So cau hinh: 90
Outer Fold: 5
Inner Fold: 3
Tong so luot fit CV: 1350



## 6. Huấn luyện và đánh giá Nested K-Fold

Với mỗi Outer Fold:

1. Chia dữ liệu thành Outer Train và Outer Test.
2. Chạy GridSearchCV trên Outer Train.
3. Tìm cấu hình có RMSE CV nhỏ nhất.
4. Huấn luyện lại mô hình tốt nhất trên toàn bộ Outer Train.
5. Dự đoán giá kim cương trên Outer Test.
6. Tính MSE, RMSE, MAE và R².
7. Lưu kết quả và bộ siêu tham số tốt nhất.

Không sử dụng Outer Test trong quá trình tìm siêu tham số.

Sau khi hoàn thành 5 Fold, tổng hợp kết quả để đánh giá mô hình.


In [6]:

import time
# Tao danh sach luu ket qua cua 5 Outer Fold
results = []

# Tao danh sach luu sieu tham so tot nhat
best_params_list = []

# Ghi nhan thoi diem bat dau toan bo thi nghiem
start_time = time.perf_counter()

# Lap qua tung Fold ben ngoai
for fold, (train_idx, test_idx) in enumerate(
    outer_kf.split(X, y), start=1
):

    # Hien thi Fold dang thuc hien
    print(f"\n========== OUTER FOLD {fold}/5 ==========", flush=True)

    # Lay dac trung cua tap Train
    X_train = X.iloc[train_idx]

    # Lay dac trung cua tap Test
    X_test = X.iloc[test_idx]

    # Lay gia thuc te cua tap Train
    y_train = y.iloc[train_idx]

    # Lay gia thuc te cua tap Test
    y_test = y.iloc[test_idx]

    # Khoi tao GridSearchCV cho Fold hien tai
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=inner_kf,
        n_jobs=2,
        refit=True,
        verbose=1
    )

    # Ghi nhan thoi diem bat dau Fold
    fold_start = time.perf_counter()

    # Tim sieu tham so tren Outer Train
# Bắt đầu đo thời gian GridSearchCV của Fold
    fit_start = time.perf_counter()

# Thực hiện tìm kiếm siêu tham số
    grid_search.fit(X_train, y_train)

# Chỉ tính thời gian của lệnh fit   
    fit_time = time.perf_counter() - fit_start

    print(f"Thời gian GridSearchCV Fold {fold}: {fit_time:.2f} giây")

    # Lay Pipeline tot nhat da duoc huan luyen lai
    best_model = grid_search.best_estimator_

    # Lay bo sieu tham so tot nhat
    best_params = grid_search.best_params_

    # Lay RMSE trung binh tot nhat cua Inner CV
    best_cv_rmse = -grid_search.best_score_

    # Du doan gia tren Outer Test
    y_pred = best_model.predict(X_test)

    # Tinh MSE tren Outer Test
    mse = mean_squared_error(y_test, y_pred)

    # Tinh RMSE tren Outer Test
    rmse = np.sqrt(mse)

    # Tinh MAE tren Outer Test
    mae = mean_absolute_error(y_test, y_pred)

    # Tinh R2 tren Outer Test
    r2 = r2_score(y_test, y_pred)

    # Tinh thoi gian thuc hien Fold
    fold_time = time.perf_counter() - fold_start

    # Luu cac chi so cua Fold
    results.append({
        "Fold": fold,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "RMSE_CV": best_cv_rmse,
        "Time_seconds": fold_time
    })

    # Luu sieu tham so tot nhat cua Fold
    best_params_list.append({
        "Fold": fold,
        **best_params
    })

    # Hien thi RMSE tren Outer Test
    print(f"RMSE Test: {rmse:.4f}")

    # Hien thi MAE tren Outer Test
    print(f"MAE Test: {mae:.4f}")

    # Hien thi R2 tren Outer Test
    print(f"R2 Test: {r2:.4f}")

    # Hien thi RMSE tot nhat cua Inner CV
    print(f"RMSE CV: {best_cv_rmse:.4f}")

    # Hien thi sieu tham so tot nhat
    print("Best params:", best_params)

    # Hien thi thoi gian chay Fold theo phut
    print(f"Thoi gian Fold: {fold_time / 60:.2f} phut", flush=True)

# Tinh tong thoi gian thuc hien
total_time = time.perf_counter() - start_time

# Thong bao hoan thanh
print("\nDA HOAN THANH 5 OUTER FOLD")

# Hien thi tong thoi gian theo phut
print(f"Tong thoi gian: {total_time / 60:.2f} phut")



========== OUTER FOLD 1/5 ==========
Fitting 3 folds for each of 90 candidates, totalling 270 fits


KeyboardInterrupt: 


## 7. Tổng hợp kết quả 5 Outer Fold

Sau khi hoàn thành Nested K-Fold:

- Hiển thị kết quả của từng Fold.
- Tính trung bình MSE, RMSE, MAE và R².
- Tính độ lệch chuẩn của các chỉ số.
- Xem mức độ ổn định của mô hình giữa các Fold.

RMSE và MAE càng thấp càng tốt. R² càng cao càng tốt.

RMSE CV là kết quả dùng để chọn siêu tham số bên trong từng Outer Fold, không phải kết quả đánh giá cuối cùng.


In [ ]:

# Chuyen danh sach ket qua thanh DataFrame
results_df = pd.DataFrame(results)

# Hien thi bang ket qua cua 5 Fold
print("KET QUA TUNG OUTER FOLD")
display(results_df.round(4))

# Cac chi so can tong hop
metric_cols = ["MSE", "RMSE", "MAE", "R2"]

# Tinh gia tri trung binh cua tung chi so
mean_scores = results_df[metric_cols].mean()

# Tinh do lech chuan cua tung chi so
std_scores = results_df[metric_cols].std(ddof=0)

# Tao bang tong hop trung binh va do lech chuan
summary_df = pd.DataFrame({
    "Mean": mean_scores,
    "Std": std_scores
})

# Hien thi bang tong hop
print("\nTONG HOP KET QUA 5 FOLD")
display(summary_df.round(4))

# Hien thi RMSE trung binh
print(f"RMSE trung binh: {mean_scores['RMSE']:.4f}")

# Hien thi MAE trung binh
print(f"MAE trung binh: {mean_scores['MAE']:.4f}")

# Hien thi R2 trung binh
print(f"R2 trung binh: {mean_scores['R2']:.4f}")

# Hien thi do lech chuan RMSE
print(f"Do lech chuan RMSE: {std_scores['RMSE']:.4f}")



## 8. Phân tích bộ siêu tham số tốt nhất

Trong Nested K-Fold, mỗi Outer Fold có thể chọn một bộ siêu tham số khác nhau.

Phân tích:
- Số cây được lựa chọn ở mỗi Fold.
- Độ sâu tối đa của cây.
- Số mẫu tối thiểu tại nút lá.
- Tỷ lệ đặc trưng được xét khi phân chia.

Nếu các Fold thường chọn cấu hình tương tự nhau, đây là một dấu hiệu cho thấy việc lựa chọn siêu tham số tương đối ổn định.


In [ ]:

# Chuyen danh sach sieu tham so thanh DataFrame
params_df = pd.DataFrame(best_params_list)

# Hien thi cau hinh tot nhat cua tung Fold
print("SIEU THAM SO TOT NHAT CUA TUNG FOLD")
display(params_df)

# Danh sach ten cac sieu tham so
param_cols = [
    "model__n_estimators",
    "model__max_depth",
    "model__min_samples_leaf",
    "model__max_features"
]

# Lap qua tung sieu tham so
for col in param_cols:

    # Hien thi ten sieu tham so
    print(f"\n{col}:")

    # Dem tan suat cac gia tri duoc lua chon
    print(params_df[col].value_counts(dropna=False))


So sánh mức 3, 4 và 5 K-Fold

In [ ]:

# Tao bang so sanh cac muc K-Fold
comparison_df = pd.DataFrame({
    "Muc": ["Muc 3", "Muc 4", "Muc 5"],
    "So cau hinh": [8, 36, 90],
    "RMSE": [
        538.1197,
        535.1354,
        mean_scores["RMSE"]
    ],
    "MAE": [
        265.0177,
        264.0373,
        mean_scores["MAE"]
    ],
    "R2": [
        0.9818,
        0.9820,
        mean_scores["R2"]
    ]
})

# Hien thi bang so sanh
print("SO SANH RANDOM FOREST K-FOLD MUC 3, 4, 5")
display(comparison_df.round(4))

# Lay RMSE cua muc 4
rmse_muc4 = 535.1354

# Lay RMSE trung binh cua muc 5
rmse_muc5 = mean_scores["RMSE"]

# Tinh chenh lech RMSE muc 5 so voi muc 4
chenh_lech = rmse_muc5 - rmse_muc4

# Tinh phan tram thay doi RMSE
phan_tram = chenh_lech / rmse_muc4 * 100

# Hien thi chenh lech RMSE
print(f"\nChenh lech RMSE: {chenh_lech:+.4f}")

# Hien thi phan tram thay doi
print(f"Thay doi RMSE: {phan_tram:+.2f}%")

# Nhan xet ket qua
if chenh_lech < 0:
    print("Muc 5 co RMSE trung binh thap hon Muc 4.")
elif chenh_lech > 0:
    print("Muc 5 co RMSE trung binh cao hon Muc 4.")
else:
    print("Muc 5 va Muc 4 co RMSE trung binh bang nhau.")
